# MiniAutoGen — Debate Multiagente com AgenticLoopRuntime

Demonstração do modo **chat livre** entre agentes usando o `AgenticLoopRuntime`.

**Cenário:** Três especialistas debatem a melhor arquitetura para um sistema de pagamentos:
- **Arquiteto** — visão sistémica, trade-offs de longo prazo
- **Dev Sénior** — pragmatismo, complexidade de implementação
- **QA Lead** — testabilidade, observabilidade, riscos

Um **router inteligente** (também LLM) decide quem fala a seguir com base no estado da conversa.

**Stack:** `AgenticLoopRuntime` + `AgentAPIDriver` + Gemini CLI via gateway ASGI

In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

import httpx


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists():
            return candidate
    return start

repo_root = find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))


def assert_gemini_cli_ready() -> dict[str, object]:
    binary = subprocess.run(
        ['bash', '-lc', 'command -v gemini'],
        capture_output=True,
        text=True,
        check=False,
    )
    gemini_path = binary.stdout.strip()
    if not gemini_path:
        raise RuntimeError('Binary `gemini` não encontrado no PATH.')

    last_error = None
    for attempt in range(1, 4):
        try:
            smoke = subprocess.run(
                ['gemini', '-m', 'gemini-2.5-flash', '--output-format', 'json'],
                input='Respond with exactly: NOTEBOOK-GEMINI-OK',
                capture_output=True,
                text=True,
                check=False,
                timeout=180,
            )
            if smoke.returncode != 0:
                last_error = f'Gemini CLI falhou no smoke test: {smoke.stderr}'
                continue
            if 'NOTEBOOK-GEMINI-OK' not in smoke.stdout:
                last_error = (
                    'Smoke test do Gemini CLI não retornou o texto esperado. '
                    f'stdout={smoke.stdout!r}'
                )
                continue
            return {
                'gemini_path': gemini_path,
                'smoke_test': 'NOTEBOOK-GEMINI-OK',
                'attempts_used': attempt,
                'stderr_excerpt': smoke.stderr.strip()[:200],
            }
        except subprocess.TimeoutExpired as exc:
            last_error = f'Timeout no smoke test do Gemini CLI após {exc.timeout} segundos.'

    raise RuntimeError(last_error or 'Falha desconhecida no smoke test do Gemini CLI.')

result = assert_gemini_cli_ready()
print(f"Repo root: {repo_root}")
print(f"Gemini CLI: {result['gemini_path']}")
print(f"Smoke test: {result['smoke_test']} (tentativas: {result['attempts_used']})")

## 1. Configuração do Driver e Sessão

In [ ]:
from gemini_cli_gateway.app import app as gateway_app
from miniautogen.backends.agentapi import AgentAPIClient, AgentAPIDriver
from miniautogen.backends.models import SendTurnRequest, StartSessionRequest

client = AgentAPIClient(
    base_url='http://gemini-gateway',
    transport=httpx.ASGITransport(app=gateway_app),
    health_endpoint=None,
    max_retry_attempts=3,
    timeout_seconds=120.0,
)
driver = AgentAPIDriver(client=client, model='gemini-2.5-flash')
session = await driver.start_session(StartSessionRequest(backend_id='gemini'))
print(f"Sessão iniciada: {session.session_id}")


async def ask_llm(system_prompt: str, user_prompt: str) -> str:
    """Helper para chamadas LLM via AgentAPIDriver."""
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': user_prompt},
    ]
    async for event in driver.send_turn(
        SendTurnRequest(session_id=session.session_id, messages=messages)
    ):
        if event.type == 'message_completed':
            return event.payload.get('text', '')
    return ''

## 2. Definição dos Agentes Conversacionais

Cada agente implementa o protocolo `ConversationalAgent` com dois métodos:
- `reply(message, context)` — responde à conversa usando o LLM com seu system prompt
- `route(conversation_history)` — analisa a conversa e decide quem fala a seguir (apenas o router usa isto)

In [ ]:
import json
from miniautogen.core.contracts.agentic_loop import RouterDecision


class LLMConversationalAgent:
    """Agente conversacional alimentado por LLM via AgentAPIDriver."""

    def __init__(self, name: str, system_prompt: str, is_router: bool = False):
        self.name = name
        self.system_prompt = system_prompt
        self.is_router = is_router
        self._participants: list[str] = []

    def set_participants(self, participants: list[str]) -> None:
        self._participants = participants

    async def reply(self, message: str, context: dict) -> str:
        prompt = (
            f"Contexto da conversa até agora:\n{message}\n\n"
            f"Turno: {context.get('turn', 0)}\n"
            f"Responda de forma concisa (máximo 3 parágrafos) mantendo seu papel."
        )
        return await ask_llm(self.system_prompt, prompt)

    async def route(self, conversation_history: list) -> RouterDecision:
        history_text = "\n".join(
            f"[{msg['sender']}]: {msg['content'][:200]}"
            for msg in conversation_history[-6:]
        )

        participants_str = ", ".join(f'"{p}"' for p in self._participants)

        prompt = (
            f"Você é o moderador de um debate técnico.\n\n"
            f"Participantes disponíveis: {participants_str}\n\n"
            f"Histórico recente:\n{history_text}\n\n"
            f"Analise o estado da conversa e decida:\n"
            f"1. Quem deve falar a seguir? (escolha entre os participantes)\n"
            f"2. A discussão já convergiu o suficiente para terminar?\n"
            f"3. Qual informação ainda falta?\n\n"
            f"Responda APENAS com JSON válido neste formato:\n"
            f'{{"current_state_summary": "...", "missing_information": "...", '
            f'"next_agent": "nome_do_agente", "terminate": false, '
            f'"stagnation_risk": 0.0}}\n\n'
            f"Se a discussão convergiu, use terminate: true e next_agent: null."
        )

        raw = await ask_llm(
            "Você é um moderador imparcial de debates técnicos. "
            "Responda APENAS com JSON válido.",
            prompt,
        )

        try:
            raw_clean = raw.strip()
            if raw_clean.startswith("```"):
                raw_clean = raw_clean.split("\n", 1)[1].rsplit("```", 1)[0].strip()
            data = json.loads(raw_clean)

            next_agent = data.get("next_agent")
            if next_agent and next_agent not in self._participants:
                next_agent = self._participants[0]

            return RouterDecision(
                current_state_summary=data.get("current_state_summary", "Em andamento"),
                missing_information=data.get("missing_information", ""),
                next_agent=next_agent,
                terminate=data.get("terminate", False),
                stagnation_risk=min(
                    max(float(data.get("stagnation_risk", 0.0)), 0.0), 1.0
                ),
            )
        except (json.JSONDecodeError, KeyError, ValueError):
            last_speaker = (
                conversation_history[-1]["sender"] if conversation_history else ""
            )
            available = [p for p in self._participants if p != last_speaker]
            return RouterDecision(
                current_state_summary="Parsing falhou, continuando round-robin",
                missing_information="Aguardando mais contribuições",
                next_agent=available[0] if available else self._participants[0],
                terminate=False,
                stagnation_risk=0.3,
            )


print("Classe LLMConversationalAgent definida")

## 3. Criação dos Agentes e Plano

Instanciamos os 3 debatedores + o router, cada um com personalidade e expertise distintas.

In [ ]:
from miniautogen.core.contracts.coordination import AgenticLoopPlan
from miniautogen.core.contracts.agentic_loop import ConversationPolicy

# Agentes debatedores
arquiteto = LLMConversationalAgent(
    name="arquiteto",
    system_prompt=(
        "Você é um Arquiteto de Software sénior. Sua perspetiva é sistémica: "
        "pensa em escalabilidade, manutenibilidade, separação de concerns, "
        "e decisões que afetam o longo prazo. Defenda sua posição com argumentos "
        "técnicos concretos. Seja respeitoso mas firme."
    ),
)

dev_senior = LLMConversationalAgent(
    name="dev_senior",
    system_prompt=(
        "Você é um Desenvolvedor Sénior pragmático com 10 anos de experiência. "
        "Sua perspetiva é de quem vai implementar: complexidade operacional, "
        "curva de aprendizagem da equipa, tempo de entrega, debugging em produção. "
        "Valoriza simplicidade e entrega. Seja direto e prático."
    ),
)

qa_lead = LLMConversationalAgent(
    name="qa_lead",
    system_prompt=(
        "Você é o QA Lead responsável pela qualidade do sistema. "
        "Sua perspetiva foca em testabilidade, observabilidade, "
        "estratégia de testes (unitários, integração, E2E), riscos de regressão, "
        "e monitoramento em produção. Levante riscos que os outros podem ignorar."
    ),
)

# Router (moderador do debate)
router = LLMConversationalAgent(
    name="router",
    system_prompt="Moderador de debate técnico.",
    is_router=True,
)

participants = ["arquiteto", "dev_senior", "qa_lead"]
router.set_participants(participants)

# Plano de execução
plan = AgenticLoopPlan(
    router_agent="router",
    participants=participants,
    policy=ConversationPolicy(
        max_turns=6,
        timeout_seconds=300.0,
        stagnation_window=3,
    ),
    goal=(
        "Decidir se o novo sistema de pagamentos deve usar "
        "microserviços ou monolito modular"
    ),
    initial_message=(
        "A empresa precisa construir um novo sistema de pagamentos que processe "
        "até 10.000 transações por dia, integre com 3 gateways (Stripe, PayPal, MBWay), "
        "e precise estar em produção em 4 meses. A equipa tem 5 desenvolvedores, "
        "2 dos quais são juniores. Qual arquitetura devemos adotar: microserviços "
        "ou monolito modular?"
    ),
)

print(f"Plano criado: {plan.goal}")
print(f"Participantes: {participants}")
print(f"Política: max_turns={plan.policy.max_turns}, timeout={plan.policy.timeout_seconds}s")

## 4. Execução do Debate

O `AgenticLoopRuntime` executa o ciclo:
1. Router analisa o histórico e decide quem fala
2. Agente selecionado recebe a conversa e responde
3. Repetir até convergência, max_turns, ou stagnation

In [ ]:
import uuid
from datetime import datetime, timezone
from miniautogen.core.runtime import AgenticLoopRuntime, PipelineRunner
from miniautogen.core.contracts import RunContext
from miniautogen.core.events import InMemoryEventSink
from miniautogen.stores import InMemoryRunStore, InMemoryCheckpointStore

# Registrar agentes
agent_registry = {
    "router": router,
    "arquiteto": arquiteto,
    "dev_senior": dev_senior,
    "qa_lead": qa_lead,
}

# Montar runtime
event_sink = InMemoryEventSink()
runner = PipelineRunner(
    event_sink=event_sink,
    run_store=InMemoryRunStore(),
    checkpoint_store=InMemoryCheckpointStore(),
)

runtime = AgenticLoopRuntime(runner=runner, agent_registry=agent_registry)

# Contexto de execução
context = RunContext(
    run_id=f"debate-{uuid.uuid4().hex[:8]}",
    started_at=datetime.now(timezone.utc),
    correlation_id=str(uuid.uuid4()),
    input_payload=plan.goal,
    timeout_seconds=plan.policy.timeout_seconds,
)

print(f"Iniciando debate: {context.run_id}")
print(f"Pergunta inicial: {plan.initial_message[:100]}...")
print("-" * 60)

# Executar
result = await runtime.run(
    agents=list(agent_registry.values()),
    context=context,
    plan=plan,
)

print(f"\nDebate finalizado!")
print(f"Status: {result.status}")
print(f"Turnos: {result.metadata.get('turns', '?')}")
print(f"Razão de parada: {result.metadata.get('stop_reason', '?')}")

## 5. Conversa Completa

Visualização formatada de toda a conversa entre os agentes.

In [ ]:
conversation = result.output or []

LABEL_MAP = {
    "arquiteto": "[ARQUITETO]",
    "dev_senior": "[DEV SENIOR]",
    "qa_lead": "[QA LEAD]",
    "system": "[SISTEMA]",
}

for i, msg in enumerate(conversation):
    sender = msg.get("sender", "?")
    content = msg.get("content", "")
    label = LABEL_MAP.get(sender, f"[{sender.upper()}]")

    print(f"\n{'='*60}")
    print(f"{label} (turno {i})")
    print(f"{'='*60}")
    print(content[:500])
    if len(content) > 500:
        print(f"... ({len(content)} chars total)")

print(f"\n{'='*60}")
print(f"Total de mensagens: {len(conversation)}")

## 6. Análise de Eventos do Runtime

O runtime emite eventos canónicos que permitem auditar cada decisão do router e cada resposta dos agentes.

In [ ]:
print(f"Total de eventos: {len(event_sink.events)}\n")

for event in event_sink.events:
    etype = event.type
    payload = event.payload

    if etype == "agentic_loop_started":
        print(f"LOOP INICIADO — router: {payload.get('router', '?')}, "
              f"participantes: {payload.get('participants', [])}")

    elif etype == "router_decision":
        print(f"ROUTER -> proximo: {payload.get('next_agent', '?')} "
              f"| terminar: {payload.get('terminate', False)}")

    elif etype == "agent_replied":
        print(f"AGENTE {payload.get('agent', '?').upper()} respondeu "
              f"({payload.get('reply_length', 0)} chars)")

    elif etype == "agentic_loop_stopped":
        print(f"LOOP PARADO — razao: {payload.get('stop_reason', '?')} "
              f"| turnos: {payload.get('turns', '?')}")

    elif etype == "stagnation_detected":
        print("STAGNACAO DETETADA")

    elif etype == "run_timed_out":
        print("TIMEOUT")

    else:
        print(f"{etype}: {str(payload)[:80]}")

## 7. Decisões do Router

Análise detalhada de como o router conduziu o debate.

In [ ]:
router_events = [e for e in event_sink.events if e.type == "router_decision"]

print(f"Decisões do router: {len(router_events)}\n")

for i, event in enumerate(router_events):
    p = event.payload
    print(f"Decisão {i + 1}:")
    print(f"  Próximo agente: {p.get('next_agent', 'N/A')}")
    print(f"  Terminar: {p.get('terminate', False)}")
    print()

## 8. Limpeza

In [ ]:
await driver.close_session(session.session_id)
await client.close()
print("Sessão e cliente encerrados.")